In [ ]:
# Import adata and load the rna data 
import anndata as ad

DATA_DIR = "/home/ubuntu/data/frangieh"

rna_data_original = ad.read_h5ad(f"{DATA_DIR}/rna.h5ad")

# X is stored as CSC (column/gene-sparse), which is slow for the repeated
# cell-based (row) slicing we do throughout QC (chunking, filtering). CSR
# is the efficient layout for that access pattern.
rna_data_original.X = rna_data_original.X.tocsr()

In [ ]:
# Load the protein data
protein_data_original = ad.read_h5ad(f"{DATA_DIR}/protein.h5ad")

In [ ]:
import scanpy as sc

# Flag mitochondrial genes (human naming convention; adjust prefix if needed)
rna_data_original.var["mt"] = rna_data_original.var_names.str.upper().str.startswith("MT-")

sc.pp.calculate_qc_metrics(
    rna_data_original, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True
)

In [ ]:
# Inspect QC metric distributions before choosing filtering thresholds
sc.pl.violin(
    rna_data_original,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
# Doublet detection (not covered by the existing precomputed QC columns).
# Run in chunks rather than on all 218k cells at once: Scrublet simulates
# synthetic doublets on top of the real cells (default sim_doublet_ratio=2.0),
# so the full run needs ~3x the cells in memory simultaneously, which was
# crashing the kernel. Chunking keeps only one chunk (+ its simulated
# doublets) in memory at a time.
import gc
import os

import numpy as np

checkpoint_path = f"{DATA_DIR}/rna_qc_checkpoint.h5ad"

if os.path.exists(checkpoint_path):
    print(f"Loading existing checkpoint from {checkpoint_path}")
    rna_data_original = ad.read_h5ad(checkpoint_path)
    rna_data_original.X = rna_data_original.X.tocsr()
else:
    n_chunks = 6
    chunks = np.array_split(np.arange(rna_data_original.n_obs), n_chunks)

    doublet_score = np.empty(rna_data_original.n_obs, dtype=float)
    predicted_doublet = np.empty(rna_data_original.n_obs, dtype=bool)

    for i, idx in enumerate(chunks):
        print(f"Scrublet chunk {i + 1}/{n_chunks} ({len(idx)} cells)")
        chunk = rna_data_original[idx].copy()
        sc.pp.scrublet(chunk)
        doublet_score[idx] = chunk.obs["doublet_score"].values
        predicted_doublet[idx] = chunk.obs["predicted_doublet"].values
        del chunk
        gc.collect()

    rna_data_original.obs["doublet_score"] = doublet_score
    rna_data_original.obs["predicted_doublet"] = predicted_doublet

rna_data_original.obs["predicted_doublet"].value_counts()

In [ ]:
# Checkpoint: persist QC metrics + doublet calls so Scrublet never needs to rerun
rna_data_original.write_h5ad(f"{DATA_DIR}/rna_qc_checkpoint.h5ad")

In [ ]:
# Conservative filtering: this data has already been QC'd upstream (see notebook discussion),
# so we only trim a floor + a mito ceiling and drop predicted doublets, rather than
# re-tightening thresholds around the bulk of the existing distribution.
# Built as a single combined mask + one copy to avoid stacking up multiple
# full-size intermediate copies in memory alongside rna_data_original.
keep = (
    ~rna_data_original.obs["predicted_doublet"]
    & (rna_data_original.obs["n_genes_by_counts"] >= 200)
    & (rna_data_original.obs["total_counts"] >= 500)
    & (rna_data_original.obs["pct_counts_mt"] < 18)
)
rna_data = rna_data_original[keep].copy()

sc.pp.filter_genes(rna_data, min_cells=3)

print(f"Cells: {rna_data_original.n_obs} -> {rna_data.n_obs}")
print(f"Genes: {rna_data_original.n_vars} -> {rna_data.n_vars}")

In [ ]:
# Per-perturbation cell count check: make sure filtering hasn't wiped out
# any single perturbation's representation
before = rna_data_original.obs["perturbation"].value_counts()
after = rna_data.obs["perturbation"].value_counts()

perturbation_counts = (
    before.to_frame("n_cells_before")
    .join(after.to_frame("n_cells_after"))
    .fillna(0)
)
perturbation_counts["retained_frac"] = (
    perturbation_counts["n_cells_after"] / perturbation_counts["n_cells_before"]
)

perturbation_counts.sort_values("retained_frac").head(20)

In [ ]:
# Checkpoint: persist the final filtered object
rna_data.write_h5ad(f"{DATA_DIR}/rna_qc_filtered.h5ad")